In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import warnings
warnings.filterwarnings('ignore')

import joblib

# ML libraries
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, roc_curve, auc, precision_recall_curve

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

# Memory optimization
tf.keras.backend.clear_session()

In [ ]:
df_new = pd.read_csv("file name", low_memory = True)

df_new.head()

In [ ]:
# feature engineering - optional 
sensor_patterns = ['Temp', 'Peak'] # name of all extracted features 
sensor_cols = [c for c in df_new.columns if any(c.startswith(p) for p in sensor_patterns)]
# chọn mấy cái sensor này


def add_features(df_new, sensor_cols):
    new_cols = []
    
    key_sensors = sensor_cols[:6]  # assuming 6 features from 1 sensor
    
    for sensor in key_sensors: 
        col = f"{sensor}_rm10"
        df_new[col] = df_new[sensor].rolling(10, min_periods = 1).mean().astype(np.float32)
        # can adjust window size more or less than 10 
        new_cols.append(col)
        col = f"{sensor}_diff"
        df[col] = df[sensor].diff().fillna(0).astype(np.float32)
        new_cols.append(col)
    return new_cols

new_features = add_features(df_new, sensor_cols)
modeling_features = sensor_cols + new_features # all are numerical features 
print(f"Total modeling features: {len(modeling_features)}")
gc,collect()

In [ ]:
preprocessor = joblib.load('preprocessor.joblib') # file lưu từ trước 
X_new_transformed = preprocessor.transform(df_new)
print(f"Data new: {X_new_transformed.shape}")

In [ ]:
# load lại model từ file
iso_forest_loaded = joblib.load('isolation_forest_model.joblib') # train từ trước

# dự đoán trên dữ liệu new 
y_pred = iso_forest.predict(X_new_transformed)

# tính tỷ lệ abnormal
n_abnormal = np.sum(y_pred == -1)
n_total = len(y_pred)
ratio_abnormal = n_abnormal / n_total

print(f"Tỷ lệ abnormal: {n_abnormal}/{n_total} ({ratio_abnormal:.2%})")

abnormal_indices = np.where(y_pred == -1)[0]
print("5 vị trí abnormal", abnormal_indices[:5])

print("5 giá trị abnormal (original):")
print(X_new_transformed.iloc[abnormal_indices[:5]])

In [ ]:
oneclass_svm = joblib.load('oneclass_svm.joblib')

y1_pred = oneclass_svm.predict(X_new_transformed)

n1_abnormal = np.sum(y1_pred == -1)
n1_total = len(y1_pred)
ratio1_abnormal = n1_abnormal / n1_total

print(f"Tỷ lệ abnormal: {n1_abnormal}/{n1_total} ({ratio1_abnormal1:.2%})")

abnormal1_indices = np.where(y1_pred == -1)[0]
print("5 vị trí abnormal", abnormal1_indices[:5])

print("5 giá trị abnormal ")
print(X_new_transformed.iloc[abnormal1_indices[:5]])

In [ ]:
# load lại từ file HDF5
lstm_ae_loaded = load_model("lstm_autoencoder.h5")   # đã lưu từ trước 

In [ ]:
print("Calculating reconstruction errors")

X_test_pred = lstm_ae_loaded.predict(X_new_transformed, batch_size = 256, verbose = 0)

mse_test = np.mean(np.square(X_new_transformed - X_test_pred), axis = (1, 2))

threshold_95 = np.percentile(mse_test, 95)
threshold_99 = np.percentile(mse_test, 99)

print(f"threshold (95%): {threshold_95:.6f}")
print(f"threshold (99%): {threshold_99:.6f}")

In [ ]:
normal_threshold_95 = 10   # change this after training normal set 

sum = sum(l > normal_threshold_95 for l in normal_threshold_95)
print(f"number of abnormal: {sum}")

In [ ]:
plt.figure(figsize=(10,6))
plt.hist(mse_test, bins=50, alpha=0.7)
plt.xlabel("Reconstruction Error (MSE)")
plt.ylabel("Count")
plt.title("Distribution of Reconstruction Error on New Set")
plt.grid(True)
plt.show()